<a href="https://colab.research.google.com/github/AlashAdi/-minikube-nodejs-app/blob/main/LAB4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==========================================
# 1. Настройка окружения и файлов проекта
# ==========================================
import json
import operator
from pathlib import Path

# Создаем структуру директорий
ROOT = Path("academic_navigator")
for folder in ["knowledge", "src", "reports"]:
  (ROOT / folder).mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. Создание продукционных правил (knowledge/rules.json)
# ==========================================
# Требование: не менее 12 правил, по 3 на каждый класс и приоритеты для конфликтов
rules = [
    # --- Класс: жоғары (Высокий риск) ---
    {
        "id": "R_HIGH_01",
        "priority": 100,
        "all": [
            {"field": "attendance", "op": "<", "value": 70},
            {"field": "gpa", "op": "<", "value": 2.3},
        ],
        "then": {"risk": "жоғары", "action": "шұғыл көмек"},
        "explanation": "Посещаемость (<70%) и GPA (<2.3) критически низкие.",
    },
    {
        "id": "R_HIGH_02",
        "priority": 95,
        "all": [
            {"field": "late_tasks", "op": ">=", "value": 6},
            {"field": "study_hours", "op": "<", "value": 5},
        ],
        "then": {"risk": "жоғары", "action": "шұғыл көмек"},
        "explanation": "Много просроченных заданий (>=6) и мало часов учебы (<5).",
    },
    {
        "id": "R_HIGH_03",
        "priority": 90,
        "all": [{"field": "gpa", "op": "<", "value": 2.0}],
        "then": {"risk": "жоғары", "action": "шұғыл көмек"},
        "explanation": "GPA меньше 2.0 указывает на высокий академический риск.",
    },
    {
        "id": "R_HIGH_04",
        "priority": 85,
        "all": [{"field": "attendance", "op": "<", "value": 50}],
        "then": {"risk": "жоғары", "action": "шұғыл көмек"},
        "explanation": "Посещаемость ниже 50%.",
    },
    # --- Класс: орта (Средний риск) ---
    {
        "id": "R_MED_01",
        "priority": 60,
        "all": [
            {"field": "late_tasks", "op": ">=", "value": 4},
            {"field": "late_tasks", "op": "<", "value": 6},
        ],
        "then": {"risk": "орта", "action": "кеңес"},
        "explanation": "Количество просроченных заданий от 4 до 5.",
    },
    {
        "id": "R_MED_02",
        "priority": 55,
        "all": [
            {"field": "gpa", "op": ">=", "value": 2.3},
            {"field": "gpa", "op": "<", "value": 3.0},
            {"field": "attendance", "op": "<", "value": 80},
        ],
        "then": {"risk": "орта", "action": "кеңес"},
        "explanation": "Средний GPA (2.3-3.0) при сниженной посещаемости (<80%).",
    },
    {
        "id": "R_MED_03",
        "priority": 50,
        "all": [
            {"field": "study_hours", "op": ">=", "value": 5},
            {"field": "study_hours", "op": "<", "value": 10},
        ],
        "then": {"risk": "орта", "action": "кеңес"},
        "explanation": "Недостаточное количество часов самостоятельной работы (5-10 ч).",
    },
    {
        "id": "R_MED_04",
        "priority": 45,
        "all": [
            {"field": "attendance", "op": ">=", "value": 70},
            {"field": "attendance", "op": "<", "value": 85},
            {"field": "scholarship", "op": "==", "value": 0},
        ],
        "then": {"risk": "орта", "action": "кеңес"},
        "explanation": "Умеренная посещаемость без стипендии.",
    },
    # --- Класс: төмен (Низкий риск) ---
    {
        "id": "R_LOW_01",
        "priority": 20,
        "all": [
            {"field": "attendance", "op": ">=", "value": 85},
            {"field": "gpa", "op": ">=", "value": 3.0},
        ],
        "then": {"risk": "төмен", "action": "қалыпты бақылау"},
        "explanation": "Высокая посещаемость (>=85%) и хороший GPA (>=3.0).",
    },
    {
        "id": "R_LOW_02",
        "priority": 15,
        "all": [
            {"field": "late_tasks", "op": "<", "value": 2},
            {"field": "study_hours", "op": ">=", "value": 15},
        ],
        "then": {"risk": "төмен", "action": "қалыпты бақылау"},
        "explanation": "Минимум задержек (<2) и активная учеба (>=15 ч).",
    },
    {
        "id": "R_LOW_03",
        "priority": 10,
        "all": [
            {"field": "gpa", "op": ">=", "value": 3.5},
            {"field": "scholarship", "op": "==", "value": 1},
        ],
        "then": {"risk": "төмен", "action": "қалыпты бақылау"},
        "explanation": "Отличный GPA (>=3.5) и наличие стипендии.",
    },
    {
        "id": "R_LOW_04",
        "priority": 5,
        "all": [
            {"field": "attendance", "op": ">=", "value": 90},
            {"field": "late_tasks", "op": "==", "value": 0},
        ],
        "then": {"risk": "төмен", "action": "қалыпты бақылау"},
        "explanation": "Идеальная посещаемость и отсутствие сбоев по срокам.",
    },
]

# Сохраняем в knowledge/rules.json
rules_path = ROOT / "knowledge/rules.json"
rules_path.write_text(
    json.dumps(rules, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Правила сохранены в {rules_path}")

# ==========================================
# 3. Разработка движка правил (src/rule_engine.py)
# ==========================================
rule_engine_code = """import json
import operator

OPS = {
    "<": operator.lt,
    "<=": operator.le,
    ">": operator.gt,
    ">=": operator.ge,
    "==": operator.eq,
    "in": lambda a, b: a in b
}

def condition_ok(facts, condition):
    field = condition["field"]
    if field not in facts:
        return False
    return OPS[condition["op"]](facts[field], condition["value"])

def evaluate_rules(facts, rules):
    fired = []
    for rule in sorted(rules, key=lambda r: r["priority"], reverse=True):
        if "all" in rule and all(condition_ok(facts, c) for c in rule["all"]):
            fired.append({
                "rule_id": rule["id"],
                "priority": rule["priority"],
                "result": rule["then"],
                "explanation": rule["explanation"]
            })
        elif "any" in rule and any(condition_ok(facts, c) for c in rule["any"]):
            fired.append({
                "rule_id": rule["id"],
                "priority": rule["priority"],
                "result": rule["then"],
                "explanation": rule["explanation"]
            })
    return fired

def choose_result(fired):
    if not fired:
        return {"status": "unknown", "action": "адам тексеруі қажет", "explanation": "Подходящих правил не найдено"}

    top_priority = fired[0]["priority"]
    top = [x for x in fired if x["priority"] == top_priority]

    # Проверка на конфликт результатов при одинаковом приоритете
    unique_results = {json.dumps(x["result"], ensure_ascii=False, sort_keys=True) for x in top}
    if len(unique_results) > 1:
        return {"status": "conflict", "candidates": top, "action": "адам тексеруі қажет", "explanation": "Конфликт правил с одинаковым приоритетом"}

    return {"status": "resolved", **top[0]}
"""

(ROOT / "src/rule_engine.py").write_text(rule_engine_code, encoding="utf-8")
print("Движок правил сохранен в src/rule_engine.py")

# ==========================================
# 4. Тестирование экспертной системы
# ==========================================
from academic_navigator.src.rule_engine import evaluate_rules, choose_result

# Сценарии для проверки
test_cases = [
    {
        "name": "Высокий риск (Сценарий 1)",
        "facts": {
            "gpa": 2.1,
            "attendance": 65,
            "late_tasks": 5,
            "study_hours": 4,
            "scholarship": 0,
        },
    },
    {
        "name": "Средний риск (Сценарий 2)",
        "facts": {
            "gpa": 2.8,
            "attendance": 78,
            "late_tasks": 4,
            "study_hours": 8,
            "scholarship": 0,
        },
    },
    {
        "name": "Низкий риск (Сценарий 3)",
        "facts": {
            "gpa": 3.8,
            "attendance": 95,
            "late_tasks": 0,
            "study_hours": 20,
            "scholarship": 1,
        },
    },
    {
        "name": "Неизвестный случай (Unknown)",
        "facts": {
            "gpa": 2.25,
            "attendance": 82,
            "late_tasks": 2,
            "study_hours": 12,
            "scholarship": 0,
        },
    },
]

print("\n--- Результаты тестирования Лабораторной №4 ---")
for case in test_cases:
  fired = evaluate_rules(case["facts"], rules)
  decision = choose_result(fired)
  print(f"\nТест: {case['name']}")
  print(f"Входные данные: {case['facts']}")
  print(f"Решение: {json.dumps(decision, ensure_ascii=False, indent=2)}")

Правила сохранены в academic_navigator/knowledge/rules.json
Движок правил сохранен в src/rule_engine.py

--- Результаты тестирования Лабораторной №4 ---

Тест: Высокий риск (Сценарий 1)
Входные данные: {'gpa': 2.1, 'attendance': 65, 'late_tasks': 5, 'study_hours': 4, 'scholarship': 0}
Решение: {
  "status": "resolved",
  "rule_id": "R_HIGH_01",
  "priority": 100,
  "result": {
    "risk": "жоғары",
    "action": "шұғыл көмек"
  },
  "explanation": "Посещаемость (<70%) и GPA (<2.3) критически низкие."
}

Тест: Средний риск (Сценарий 2)
Входные данные: {'gpa': 2.8, 'attendance': 78, 'late_tasks': 4, 'study_hours': 8, 'scholarship': 0}
Решение: {
  "status": "resolved",
  "rule_id": "R_MED_01",
  "priority": 60,
  "result": {
    "risk": "орта",
    "action": "кеңес"
  },
  "explanation": "Количество просроченных заданий от 4 до 5."
}

Тест: Низкий риск (Сценарий 3)
Входные данные: {'gpa': 3.8, 'attendance': 95, 'late_tasks': 0, 'study_hours': 20, 'scholarship': 1}
Решение: {
  "status": 

In [8]:
from pathlib import Path

ROOT = Path("academic_navigator")
required_files = [
    ROOT / "knowledge/rules.json",
    ROOT / "src/rule_engine.py"
]

for path in required_files:
    if path.exists() and path.stat().st_size > 0:
        print(f"✅ Файл найден и содержить данные : {path} (Размер: {path.stat().st_size} байт)")
    else:
        print(f"❌ Ошибка! Файл отсутствует или пуст: {path}")

✅ Файл найден и содержить данные : academic_navigator/knowledge/rules.json (Размер: 5202 байт)
✅ Файл найден и содержить данные : academic_navigator/src/rule_engine.py (Размер: 1913 байт)


In [9]:
import json
from pathlib import Path
from academic_navigator.src.rule_engine import evaluate_rules, choose_result

ROOT = Path("academic_navigator")
rules = json.loads((ROOT / "knowledge/rules.json").read_text(encoding="utf-8"))

# Набор тестовых студентов (фактов)
test_students = [
    {
        "name": "Студент 1 (Ожидается: Высокий риск / шұғыл көмек)",
        "facts": {"gpa": 2.1, "attendance": 65, "late_tasks": 5, "study_hours": 4, "scholarship": 0}
    },
    {
        "name": "Студент 2 (Ожидается: Средний риск / кеңес)",
        "facts": {"gpa": 2.8, "attendance": 78, "late_tasks": 4, "study_hours": 8, "scholarship": 0}
    },
    {
        "name": "Студент 3 (Ожидается: Низкий риск / қалыпты бақылау)",
        "facts": {"gpa": 3.8, "attendance": 95, "late_tasks": 0, "study_hours": 20, "scholarship": 1}
    },
    {
        "name": "Студент 4 (Ожидается: Unknown / Неизвестно)",
        "facts": {"gpa": 2.25, "attendance": 82, "late_tasks": 2, "study_hours": 12, "scholarship": 0}
    }
]

print("=== ТЕСТИРОВАНИЕ ЭКСПЕРТНОЙ СИСТЕМЫ ===\n")
for test in test_students:
    fired = evaluate_rules(test["facts"], rules)
    decision = choose_result(fired)

    print(f"📌 {test['name']}")
    print(f"   Входные данные: {test['facts']}")
    print(f"   Статус: {decision.get('status')}")
    if decision.get("status") == "resolved":
        print(f"   Правило ID: {decision.get('rule_id')} (Приоритет: {decision.get('priority')})")
        print(f"   Результат: {decision.get('result')}")
        print(f"   Объяснение: {decision.get('explanation')}")
    else:
        print(f"   Действие: {decision.get('action')}")
        print(f"   Объяснение: {decision.get('explanation')}")
    print("-" * 60)

=== ТЕСТИРОВАНИЕ ЭКСПЕРТНОЙ СИСТЕМЫ ===

📌 Студент 1 (Ожидается: Высокий риск / шұғыл көмек)
   Входные данные: {'gpa': 2.1, 'attendance': 65, 'late_tasks': 5, 'study_hours': 4, 'scholarship': 0}
   Статус: resolved
   Правило ID: R_HIGH_01 (Приоритет: 100)
   Результат: {'risk': 'жоғары', 'action': 'шұғыл көмек'}
   Объяснение: Посещаемость (<70%) и GPA (<2.3) критически низкие.
------------------------------------------------------------
📌 Студент 2 (Ожидается: Средний риск / кеңес)
   Входные данные: {'gpa': 2.8, 'attendance': 78, 'late_tasks': 4, 'study_hours': 8, 'scholarship': 0}
   Статус: resolved
   Правило ID: R_MED_01 (Приоритет: 60)
   Результат: {'risk': 'орта', 'action': 'кеңес'}
   Объяснение: Количество просроченных заданий от 4 до 5.
------------------------------------------------------------
📌 Студент 3 (Ожидается: Низкий риск / қалыпты бақылау)
   Входные данные: {'gpa': 3.8, 'attendance': 95, 'late_tasks': 0, 'study_hours': 20, 'scholarship': 1}
   Статус: resolved

In [11]:
def validate_rules(rules):
    assert len(rules) >= 12, f"Должно быть минимум 12 правил, сейчас: {len(rules)}"
    rule_ids = [r["id"] for r in rules]
    assert len(rule_ids) == len(set(rule_ids)), "Есть дублирующиеся rule_id!"

    for r in rules:
        assert "id" in r and "priority" in r and "then" in r and "explanation" in r
        assert "risk" in r["then"] and "action" in r["then"]
        assert "all" in r or "any" in r

    print("✅ Валидация правил прошла успешно! База знаний корректна.")

validate_rules(rules)

✅ Валидация правил прошла успешно! База знаний корректна.
